In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

import patch_dataset
importlib.reload(patch_dataset)
from patch_dataset import (
    extract_patch_dataset, extract_registered_pair, prepare_patch_positions
)

In [ ]:
DATA_ROOT = Path('../../data/HnE_n_UNStaining')
REGISTRATION_DIR = DATA_ROOT / 'registration_results' / 'matrices'
OUTPUT_DIR = DATA_ROOT / 'patch_dataset_mpp05_2048'
PREVIEW_DIR = DATA_ROOT / 'patch_dataset_preview_mpp05'

TARGET_MPP = 0.5
SOURCE_MPP = 0.25  # 모든 변환 TIFF를 native 0.5 MPP로 고정
PATCH_SIZE = 2048
OVERLAP = 0.2
TISSUE_THRESHOLD = 0.3
RUN_FULL_EXTRACTION = True

registration_jsons = sorted(REGISTRATION_DIR.glob('*_registration.json'))
print(f'Registration files: {len(registration_jsons)}')

In [ ]:
estimated_counts = []
for json_path in registration_jsons:
    _, positions, patch_size_wsi, stride_wsi, mpp, used_fallback = prepare_patch_positions(
        json_path,
        target_mpp=TARGET_MPP,
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        tissue_threshold=TISSUE_THRESHOLD,
        fallback_mpp=SOURCE_MPP,
    )
    estimated_counts.append((json_path.name, len(positions)))

print(f'Estimated paired patches: {sum(count for _, count in estimated_counts)}')
print('Slides with zero patches:', [name for name, count in estimated_counts if count == 0])

In [ ]:
PREVIEW_JSON_IDX = 4
preview_json = registration_jsons[PREVIEW_JSON_IDX]
preview_rows, preview_summary = extract_registered_pair(
    preview_json,
    PREVIEW_DIR,
    target_mpp=TARGET_MPP,
    patch_size=PATCH_SIZE,
    overlap=OVERLAP,
    tissue_threshold=TISSUE_THRESHOLD,
    fallback_mpp=SOURCE_MPP,
    max_patches=5,
    overwrite=False,
)
print(preview_summary)

fig, axes = plt.subplots(len(preview_rows), 2, figsize=(10, 5 * len(preview_rows)))
if len(preview_rows) == 1:
    axes = axes[None, :]
for row_axes, row in zip(axes, preview_rows):
    name = row['patch_name']
    row_axes[0].imshow(Image.open(PREVIEW_DIR / 'unstain' / name))
    row_axes[0].set_title(f'Unstain: {name}')
    row_axes[1].imshow(Image.open(PREVIEW_DIR / 'hne' / name))
    row_axes[1].set_title(f'Aligned H&E: {name}')
    for axis in row_axes:
        axis.axis('off')
plt.tight_layout()

In [ ]:
if RUN_FULL_EXTRACTION:
    extraction_summary = extract_patch_dataset(
        registration_jsons,
        OUTPUT_DIR,
        target_mpp=TARGET_MPP,
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        tissue_threshold=TISSUE_THRESHOLD,
        fallback_mpp=SOURCE_MPP,
        overwrite=False,
    )
    errors = [row for row in extraction_summary if row['status'] == 'error']
    print(f'Completed slides: {len(extraction_summary) - len(errors)}')
    print(f'Errors: {len(errors)}')
else:
    print('Preview를 확인한 뒤 RUN_FULL_EXTRACTION=True로 변경하세요.')